In [ ]:
# Lecture 3 — What Apache Spark Is
# Spark 101 · Module 1 · Lecture 3
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.
#
# Demo: word count in PySpark. The same job that took ~100 lines of Java in
# MapReduce fits in ~10 lines here — and reads like SQL or pandas.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- Get the input text ----------------------------------------------------
# Locally: run download_shakespeare.sh (or the file sits next to this notebook).
# On Databricks: upload shakespeare.txt, or fetch it into /tmp as below.
import os, urllib.request

path = "/tmp/spark101_l03/shakespeare.txt"
os.makedirs(os.path.dirname(path), exist_ok=True)
if not os.path.exists(path):
    url = "https://www.gutenberg.org/files/100/100-0.txt"  # complete works
    urllib.request.urlretrieve(url, path)
print("input:", path, "-", os.path.getsize(path) // 1024, "KB")

In [ ]:
# --- Word count — the whole job, ~10 lines --------------------------------
from pyspark.sql.functions import col, count, explode, lower, split

lines = spark.read.text(f"file:{path}")

word_counts = (
    lines.select(explode(split(lower(col("value")), r"\W+")).alias("word"))
         .filter(col("word") != "")
         .groupBy("word")
         .agg(count("*").alias("n"))
         .orderBy(col("n").desc())
)

In [ ]:
# --- Trigger it: show the top 20 words ------------------------------------
print("Top 20 words in the complete works of Shakespeare:")
word_counts.show(20, truncate=False)
print("distinct words:", word_counts.count())

In [ ]:
# Only "real" words: keep tokens longer than 3 characters.
from pyspark.sql.functions import length
word_counts.filter(length("word") > 3).show(20, truncate=False)

In [ ]:
# Totals: how many words in all, and how many distinct?
from pyspark.sql.functions import sum as _sum
print("total words:", word_counts.agg(_sum("n")).first()[0])
print("distinct words:", word_counts.count())

In [ ]:
# It's just a DataFrame — look up one word's count.
word_counts.filter(word_counts.word == "the").show()

In [ ]:
# Recap:
#   - Every line above is one transformation — reads like SQL / pandas.
#   - No SparkSession boilerplate on Databricks; `spark` is provided.
#   - This ~10-line job replaces ~100 lines of Java MapReduce — and runs in memory.
# Next module: the environment (Databricks) and your first program from scratch.